# 05 — End-to-End Data Quality Checks

## Banking Reporting Platform

This notebook is the final validation step for the data pipeline before reporting in Metabase.

Pipeline under test:

```text
CSV source files
        ↓
      raw
        ↓
    staging
        ↓
      marts
        ↓
    Metabase
```

The goal is to confirm that the complete database is internally consistent and ready for reporting.

This notebook checks:

- row-count reconciliation
- business-key uniqueness
- referential integrity
- staging business rules
- mart dimensional integrity
- fact-table grain
- bridge allocation
- SCD Type 1 design
- KPI sanity
- reporting readiness

Unlike notebook 01, which inspected incoming source files, this notebook validates the **completed database pipeline**.


## 1. Imports and Database Configuration


In [1]:
from pathlib import Path

import pandas as pd
import psycopg
from psycopg import sql
from IPython.display import display


def find_project_root(start: Path) -> Path:
    start = start.resolve()

    for candidate in [start, *start.parents]:
        if (candidate / "data" / "raw").exists():
            return candidate

    raise FileNotFoundError(
        "Could not locate the project root. Expected a data/raw directory "
        "in the current directory or one of its parents."
    )


def read_env_file(path: Path) -> dict[str, str]:
    if not path.exists():
        raise FileNotFoundError(
            f"{path} was not found. Create .env from .env.example before continuing."
        )

    values = {}

    for raw_line in path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()

        if not line or line.startswith("#") or "=" not in line:
            continue

        key, value = line.split("=", 1)
        values[key.strip()] = value.strip().strip('"').strip("'")

    return values


PROJECT_ROOT = find_project_root(Path.cwd())
ENV_PATH = PROJECT_ROOT / ".env"

env = read_env_file(ENV_PATH)

DB_CONFIG = {
    "host": "localhost",
    "port": int(env["POSTGRES_PORT"]),
    "dbname": env["POSTGRES_DB"],
    "user": env["POSTGRES_USER"],
    "password": env["POSTGRES_PASSWORD"],
}


def get_connection():
    return psycopg.connect(**DB_CONFIG)


def query_dataframe(query: str, params=None) -> pd.DataFrame:
    with get_connection() as conn:
        with conn.cursor() as cur:
            cur.execute(query, params)
            columns = [description.name for description in cur.description]
            rows = cur.fetchall()

    return pd.DataFrame(rows, columns=columns)


print(f"Project root: {PROJECT_ROOT}")
print(
    "PostgreSQL target:",
    f"{DB_CONFIG['user']}@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['dbname']}",
)


Project root: C:\Users\Admin\Projects\banking-reporting-platform
PostgreSQL target: banking_admin@localhost:5432/analytics


## 2. Confirm Required Tables Exist

All expected raw, staging, audit, and mart tables must be present before quality checks begin.


In [2]:
EXPECTED_TABLES = {
    "raw": {
        "customers",
        "accounts",
        "customer_accounts",
        "transactions",
    },
    "staging": {
        "customers",
        "accounts",
        "customer_accounts",
        "channels",
        "transactions",
    },
    "audit": {
        "rejected_records",
    },
    "marts": {
        "dim_customer",
        "dim_account",
        "dim_channel",
        "dim_date",
        "bridge_account_customer",
        "fact_transactions",
    },
}

table_inventory = query_dataframe(
    '''
    SELECT
        table_schema,
        table_name
    FROM information_schema.tables
    WHERE table_schema IN ('raw', 'staging', 'audit', 'marts')
      AND table_type = 'BASE TABLE'
    ORDER BY table_schema, table_name;
    '''
)

display(table_inventory)

actual_tables = {
    schema: set(
        table_inventory.loc[
            table_inventory["table_schema"].eq(schema),
            "table_name",
        ]
    )
    for schema in EXPECTED_TABLES
}

missing_tables = []

for schema, expected in EXPECTED_TABLES.items():
    for table_name in sorted(expected - actual_tables.get(schema, set())):
        missing_tables.append(f"{schema}.{table_name}")

if missing_tables:
    raise RuntimeError(f"Missing required tables: {missing_tables}")

print("All required tables are present.")


,table_schema,table_name
0,audit,rejected_records
1,marts,bridge_account_customer
2,marts,dim_account
3,marts,dim_channel
4,marts,dim_customer
5,marts,dim_date
6,marts,fact_transactions
7,raw,accounts
8,raw,customer_accounts
9,raw,customers


All required tables are present.


## 3. Row Counts Across the Pipeline

This gives a quick view of the size of each layer.


In [3]:
TABLES_TO_COUNT = [
    ("raw", "customers"),
    ("raw", "accounts"),
    ("raw", "customer_accounts"),
    ("raw", "transactions"),
    ("staging", "customers"),
    ("staging", "accounts"),
    ("staging", "customer_accounts"),
    ("staging", "channels"),
    ("staging", "transactions"),
    ("audit", "rejected_records"),
    ("marts", "dim_customer"),
    ("marts", "dim_account"),
    ("marts", "dim_channel"),
    ("marts", "dim_date"),
    ("marts", "bridge_account_customer"),
    ("marts", "fact_transactions"),
]

row_counts = []

with get_connection() as conn:
    with conn.cursor() as cur:
        for schema_name, table_name in TABLES_TO_COUNT:
            cur.execute(
                sql.SQL("SELECT COUNT(*) FROM {}.{}").format(
                    sql.Identifier(schema_name),
                    sql.Identifier(table_name),
                )
            )

            row_counts.append(
                {
                    "table": f"{schema_name}.{table_name}",
                    "rows": cur.fetchone()[0],
                }
            )

row_counts = pd.DataFrame(row_counts)
display(row_counts)


,table,rows
0,raw.customers,1003
1,raw.accounts,1253
2,raw.customer_accounts,1355
3,raw.transactions,50010
4,staging.customers,1000
5,staging.accounts,1250
6,staging.customer_accounts,1350
7,staging.channels,3
8,staging.transactions,50000
9,audit.rejected_records,21


## 4. Raw-to-Staging Reconciliation

For the four source entities:

```text
raw rows = staging accepted rows + audit rejected rows
```

This confirms that source records were neither silently lost nor duplicated during validation.


In [4]:
raw_to_staging = query_dataframe(
    '''
    WITH rejected AS (
        SELECT
            source_name,
            COUNT(*) AS rejected_rows
        FROM audit.rejected_records
        GROUP BY source_name
    )
    SELECT
        source_name,
        raw_rows,
        staging_rows,
        COALESCE(rejected_rows, 0) AS rejected_rows,
        raw_rows - staging_rows - COALESCE(rejected_rows, 0) AS difference
    FROM (
        SELECT
            'customers' AS source_name,
            (SELECT COUNT(*) FROM raw.customers) AS raw_rows,
            (SELECT COUNT(*) FROM staging.customers) AS staging_rows

        UNION ALL

        SELECT
            'accounts',
            (SELECT COUNT(*) FROM raw.accounts),
            (SELECT COUNT(*) FROM staging.accounts)

        UNION ALL

        SELECT
            'customer_accounts',
            (SELECT COUNT(*) FROM raw.customer_accounts),
            (SELECT COUNT(*) FROM staging.customer_accounts)

        UNION ALL

        SELECT
            'transactions',
            (SELECT COUNT(*) FROM raw.transactions),
            (SELECT COUNT(*) FROM staging.transactions)
    ) counts
    LEFT JOIN rejected
        USING (source_name)
    ORDER BY source_name;
    '''
)

raw_to_staging["passed"] = raw_to_staging["difference"].eq(0)
display(raw_to_staging)

if not raw_to_staging["passed"].all():
    raise RuntimeError("Raw-to-staging reconciliation failed.")

print("Raw-to-staging reconciliation passed.")


,source_name,raw_rows,staging_rows,rejected_rows,difference,passed
0,accounts,1253,1250,3,0,True
1,customer_accounts,1355,1350,5,0,True
2,customers,1003,1000,3,0,True
3,transactions,50010,50000,10,0,True


Raw-to-staging reconciliation passed.


## 5. Staging Business-Key Uniqueness

The staging layer should contain only one accepted row per approved business key.


In [5]:
staging_key_checks = query_dataframe(
    '''
    SELECT
        'staging.customers.customer_id' AS check_name,
        COUNT(*) AS failed_rows
    FROM (
        SELECT customer_id
        FROM staging.customers
        GROUP BY customer_id
        HAVING COUNT(*) > 1
    ) duplicates

    UNION ALL

    SELECT
        'staging.accounts.account_id',
        COUNT(*)
    FROM (
        SELECT account_id
        FROM staging.accounts
        GROUP BY account_id
        HAVING COUNT(*) > 1
    ) duplicates

    UNION ALL

    SELECT
        'staging.customer_accounts(customer_id, account_id)',
        COUNT(*)
    FROM (
        SELECT customer_id, account_id
        FROM staging.customer_accounts
        GROUP BY customer_id, account_id
        HAVING COUNT(*) > 1
    ) duplicates

    UNION ALL

    SELECT
        'staging.transactions.transaction_id',
        COUNT(*)
    FROM (
        SELECT transaction_id
        FROM staging.transactions
        GROUP BY transaction_id
        HAVING COUNT(*) > 1
    ) duplicates;
    '''
)

staging_key_checks["passed"] = staging_key_checks["failed_rows"].eq(0)
display(staging_key_checks)

if not staging_key_checks["passed"].all():
    raise RuntimeError("Staging business-key uniqueness failed.")

print("All staging business keys are unique.")


,check_name,failed_rows,passed
0,staging.customers.customer_id,0,True
1,staging.accounts.account_id,0,True
2,"staging.customer_accounts(customer_id, account...",0,True
3,staging.transactions.transaction_id,0,True


All staging business keys are unique.


## 6. Staging Referential Integrity

The database foreign keys enforce these relationships, but explicit validation makes the pipeline checks visible and auditable.


In [6]:
staging_fk_checks = query_dataframe(
    '''
    SELECT
        'customer_accounts -> customers' AS check_name,
        COUNT(*) AS failed_rows
    FROM staging.customer_accounts ca
    LEFT JOIN staging.customers c
        ON c.customer_id = ca.customer_id
    WHERE c.customer_id IS NULL

    UNION ALL

    SELECT
        'customer_accounts -> accounts',
        COUNT(*)
    FROM staging.customer_accounts ca
    LEFT JOIN staging.accounts a
        ON a.account_id = ca.account_id
    WHERE a.account_id IS NULL

    UNION ALL

    SELECT
        'transactions -> accounts',
        COUNT(*)
    FROM staging.transactions t
    LEFT JOIN staging.accounts a
        ON a.account_id = t.account_id
    WHERE a.account_id IS NULL

    UNION ALL

    SELECT
        'transactions -> channels',
        COUNT(*)
    FROM staging.transactions t
    LEFT JOIN staging.channels c
        ON c.channel_code = t.channel_code
    WHERE c.channel_code IS NULL;
    '''
)

staging_fk_checks["passed"] = staging_fk_checks["failed_rows"].eq(0)
display(staging_fk_checks)

if not staging_fk_checks["passed"].all():
    raise RuntimeError("Staging referential-integrity checks failed.")

print("All staging relationships are valid.")


,check_name,failed_rows,passed
0,customer_accounts -> customers,0,True
1,customer_accounts -> accounts,0,True
2,transactions -> channels,0,True
3,transactions -> accounts,0,True


All staging relationships are valid.


## 7. Staging Business Rules

These checks cover rules that go beyond simple keys.


In [7]:
staging_business_rules = query_dataframe(
    '''
    SELECT
        'every account has at least one holder' AS check_name,
        COUNT(*) AS failed_rows
    FROM staging.accounts a
    LEFT JOIN staging.customer_accounts ca
        ON ca.account_id = a.account_id
    WHERE ca.account_id IS NULL

    UNION ALL

    SELECT
        'every account has exactly one PRIMARY holder',
        COUNT(*)
    FROM (
        SELECT
            a.account_id
        FROM staging.accounts a
        LEFT JOIN staging.customer_accounts ca
            ON ca.account_id = a.account_id
        GROUP BY a.account_id
        HAVING COUNT(*) FILTER (
            WHERE ca.holder_role = 'PRIMARY'
        ) <> 1
    ) problems

    UNION ALL

    SELECT
        'closed_date is not before opened_date',
        COUNT(*)
    FROM staging.accounts
    WHERE closed_date IS NOT NULL
      AND closed_date < opened_date

    UNION ALL

    SELECT
        'CLOSED accounts have closed_date',
        COUNT(*)
    FROM staging.accounts
    WHERE account_status = 'CLOSED'
      AND closed_date IS NULL

    UNION ALL

    SELECT
        'transaction amount > 0',
        COUNT(*)
    FROM staging.transactions
    WHERE amount <= 0

    UNION ALL

    SELECT
        'transactions not before account opening',
        COUNT(*)
    FROM staging.transactions t
    JOIN staging.accounts a
        ON a.account_id = t.account_id
    WHERE t.transaction_timestamp::date < a.opened_date

    UNION ALL

    SELECT
        'transactions not after account closure',
        COUNT(*)
    FROM staging.transactions t
    JOIN staging.accounts a
        ON a.account_id = t.account_id
    WHERE a.closed_date IS NOT NULL
      AND t.transaction_timestamp::date > a.closed_date;
    '''
)

staging_business_rules["passed"] = staging_business_rules["failed_rows"].eq(0)
display(staging_business_rules)

if not staging_business_rules["passed"].all():
    raise RuntimeError("One or more staging business rules failed.")

print("All staging business rules passed.")


,check_name,failed_rows,passed
0,closed_date is not before opened_date,0,True
1,CLOSED accounts have closed_date,0,True
2,every account has at least one holder,0,True
3,every account has exactly one PRIMARY holder,0,True
4,transaction amount > 0,0,True
5,transactions not after account closure,0,True
6,transactions not before account opening,0,True


All staging business rules passed.


## 8. Staging Domain Checks

Categorical values must stay within the domains approved in the data model.


In [8]:
domain_checks = query_dataframe(
    '''
    SELECT
        'customer_status' AS check_name,
        COUNT(*) AS failed_rows
    FROM staging.customers
    WHERE customer_status NOT IN ('ACTIVE', 'INACTIVE')

    UNION ALL

    SELECT
        'account_type',
        COUNT(*)
    FROM staging.accounts
    WHERE account_type NOT IN ('TRANSACTION', 'SAVINGS')

    UNION ALL

    SELECT
        'account_status',
        COUNT(*)
    FROM staging.accounts
    WHERE account_status NOT IN ('ACTIVE', 'DORMANT', 'CLOSED')

    UNION ALL

    SELECT
        'holder_role',
        COUNT(*)
    FROM staging.customer_accounts
    WHERE holder_role NOT IN ('PRIMARY', 'JOINT')

    UNION ALL

    SELECT
        'transaction_type',
        COUNT(*)
    FROM staging.transactions
    WHERE transaction_type NOT IN (
        'PURCHASE',
        'WITHDRAWAL',
        'DEPOSIT',
        'TRANSFER'
    )

    UNION ALL

    SELECT
        'channel_code',
        COUNT(*)
    FROM staging.transactions
    WHERE channel_code NOT IN ('APP', 'ATM', 'CARD')

    UNION ALL

    SELECT
        'transaction status',
        COUNT(*)
    FROM staging.transactions
    WHERE status NOT IN ('SUCCESSFUL', 'FAILED', 'REVERSED')

    UNION ALL

    SELECT
        'currency_code',
        COUNT(*)
    FROM staging.transactions
    WHERE currency_code <> 'ZAR';
    '''
)

domain_checks["passed"] = domain_checks["failed_rows"].eq(0)
display(domain_checks)

if not domain_checks["passed"].all():
    raise RuntimeError("One or more staging domain checks failed.")

print("All staging domain checks passed.")


,check_name,failed_rows,passed
0,customer_status,0,True
1,account_type,0,True
2,holder_role,0,True
3,account_status,0,True
4,channel_code,0,True
5,currency_code,0,True
6,transaction status,0,True
7,transaction_type,0,True


All staging domain checks passed.


## 9. Staging-to-Mart Reconciliation

The current validated staging snapshot must be fully represented in the reporting model.

For dimensions, we test coverage by business key.

For bridge and fact tables, row counts should match staging directly.


In [9]:
staging_to_mart = query_dataframe(
    '''
    SELECT
        'customers represented in dim_customer' AS check_name,
        (SELECT COUNT(*) FROM staging.customers) AS staging_rows,
        (
            SELECT COUNT(*)
            FROM staging.customers s
            JOIN marts.dim_customer d
                ON d.customer_id = s.customer_id
        ) AS mart_rows

    UNION ALL

    SELECT
        'accounts represented in dim_account',
        (SELECT COUNT(*) FROM staging.accounts),
        (
            SELECT COUNT(*)
            FROM staging.accounts s
            JOIN marts.dim_account d
                ON d.account_id = s.account_id
        )

    UNION ALL

    SELECT
        'channels represented in dim_channel',
        (SELECT COUNT(*) FROM staging.channels),
        (
            SELECT COUNT(*)
            FROM staging.channels s
            JOIN marts.dim_channel d
                ON d.channel_code = s.channel_code
        )

    UNION ALL

    SELECT
        'customer-account relationships in bridge',
        (SELECT COUNT(*) FROM staging.customer_accounts),
        (SELECT COUNT(*) FROM marts.bridge_account_customer)

    UNION ALL

    SELECT
        'transactions in fact',
        (SELECT COUNT(*) FROM staging.transactions),
        (SELECT COUNT(*) FROM marts.fact_transactions);
    '''
)

staging_to_mart["difference"] = (
    staging_to_mart["mart_rows"] - staging_to_mart["staging_rows"]
)
staging_to_mart["passed"] = staging_to_mart["difference"].eq(0)

display(staging_to_mart)

if not staging_to_mart["passed"].all():
    raise RuntimeError("Staging-to-mart reconciliation failed.")

print("Current staging data is fully represented in marts.")


,check_name,staging_rows,mart_rows,difference,passed
0,customers represented in dim_customer,1000,1000,0,True
1,accounts represented in dim_account,1250,1250,0,True
2,channels represented in dim_channel,3,3,0,True
3,customer-account relationships in bridge,1350,1350,0,True
4,transactions in fact,50000,50000,0,True


Current staging data is fully represented in marts.


## 10. Dimension Business-Key Uniqueness

Surrogate keys are unique by primary-key constraint. This check confirms that each business key also appears only once in the current Type 1 dimensions.


In [10]:
dimension_key_checks = query_dataframe(
    '''
    SELECT
        'dim_customer.customer_id' AS check_name,
        COUNT(*) AS failed_rows
    FROM (
        SELECT customer_id
        FROM marts.dim_customer
        GROUP BY customer_id
        HAVING COUNT(*) > 1
    ) duplicates

    UNION ALL

    SELECT
        'dim_account.account_id',
        COUNT(*)
    FROM (
        SELECT account_id
        FROM marts.dim_account
        GROUP BY account_id
        HAVING COUNT(*) > 1
    ) duplicates

    UNION ALL

    SELECT
        'dim_channel.channel_code',
        COUNT(*)
    FROM (
        SELECT channel_code
        FROM marts.dim_channel
        GROUP BY channel_code
        HAVING COUNT(*) > 1
    ) duplicates

    UNION ALL

    SELECT
        'dim_date.full_date',
        COUNT(*)
    FROM (
        SELECT full_date
        FROM marts.dim_date
        GROUP BY full_date
        HAVING COUNT(*) > 1
    ) duplicates;
    '''
)

dimension_key_checks["passed"] = dimension_key_checks["failed_rows"].eq(0)
display(dimension_key_checks)

if not dimension_key_checks["passed"].all():
    raise RuntimeError("Dimension business-key uniqueness failed.")

print("All dimension business keys are unique.")


,check_name,failed_rows,passed
0,dim_customer.customer_id,0,True
1,dim_account.account_id,0,True
2,dim_channel.channel_code,0,True
3,dim_date.full_date,0,True


All dimension business keys are unique.


## 11. SCD Type 1 Design Check

The approved model uses SCD Type 1 for:

```text
marts.dim_customer
marts.dim_account
```

Therefore these dimensions should represent the current state only and should **not** contain Type 2 history-management fields such as:

```text
valid_from
valid_to
is_current
version_number
```


In [11]:
type2_style_columns = query_dataframe(
    '''
    SELECT
        table_name,
        column_name
    FROM information_schema.columns
    WHERE table_schema = 'marts'
      AND table_name IN ('dim_customer', 'dim_account')
      AND column_name IN (
          'valid_from',
          'valid_to',
          'is_current',
          'version_number'
      )
    ORDER BY table_name, column_name;
    '''
)

display(type2_style_columns)

if not type2_style_columns.empty:
    raise RuntimeError(
        "Unexpected SCD Type 2-style columns were found in Type 1 dimensions."
    )

print("SCD Type 1 physical design is consistent with the approved model.")


,table_name,column_name


SCD Type 1 physical design is consistent with the approved model.


## 12. Fact-Table Grain and Referential Integrity

Declared fact grain:

> **One row per validated financial transaction.**

Each fact row must also resolve to a valid Account, Channel, and Date dimension row.


In [12]:
fact_checks = query_dataframe(
    '''
    SELECT
        'duplicate transaction_id' AS check_name,
        COUNT(*) AS failed_rows
    FROM (
        SELECT transaction_id
        FROM marts.fact_transactions
        GROUP BY transaction_id
        HAVING COUNT(*) > 1
    ) duplicates

    UNION ALL

    SELECT
        'fact -> dim_account',
        COUNT(*)
    FROM marts.fact_transactions f
    LEFT JOIN marts.dim_account a
        ON a.account_key = f.account_key
    WHERE a.account_key IS NULL

    UNION ALL

    SELECT
        'fact -> dim_channel',
        COUNT(*)
    FROM marts.fact_transactions f
    LEFT JOIN marts.dim_channel c
        ON c.channel_key = f.channel_key
    WHERE c.channel_key IS NULL

    UNION ALL

    SELECT
        'fact -> dim_date',
        COUNT(*)
    FROM marts.fact_transactions f
    LEFT JOIN marts.dim_date d
        ON d.date_key = f.date_key
    WHERE d.date_key IS NULL;
    '''
)

fact_checks["passed"] = fact_checks["failed_rows"].eq(0)
display(fact_checks)

if not fact_checks["passed"].all():
    raise RuntimeError("Fact-table grain or referential integrity failed.")

print("Fact grain and foreign-key relationships passed.")


,check_name,failed_rows,passed
0,fact -> dim_channel,0,True
1,fact -> dim_account,0,True
2,fact -> dim_date,0,True
3,duplicate transaction_id,0,True


Fact grain and foreign-key relationships passed.


## 13. Bridge Integrity

The bridge must:

- contain valid account keys
- contain valid customer keys
- have one row per account-customer pair
- allocate approximately 100% of each account across its holders


In [13]:
bridge_checks = query_dataframe(
    '''
    SELECT
        'bridge -> dim_account' AS check_name,
        COUNT(*) AS failed_rows
    FROM marts.bridge_account_customer b
    LEFT JOIN marts.dim_account a
        ON a.account_key = b.account_key
    WHERE a.account_key IS NULL

    UNION ALL

    SELECT
        'bridge -> dim_customer',
        COUNT(*)
    FROM marts.bridge_account_customer b
    LEFT JOIN marts.dim_customer c
        ON c.customer_key = b.customer_key
    WHERE c.customer_key IS NULL

    UNION ALL

    SELECT
        'duplicate bridge account-customer pair',
        COUNT(*)
    FROM (
        SELECT account_key, customer_key
        FROM marts.bridge_account_customer
        GROUP BY account_key, customer_key
        HAVING COUNT(*) > 1
    ) duplicates

    UNION ALL

    SELECT
        'account allocation weights sum to 1',
        COUNT(*)
    FROM (
        SELECT
            account_key,
            SUM(allocation_weight) AS allocation_total
        FROM marts.bridge_account_customer
        GROUP BY account_key
        HAVING ABS(SUM(allocation_weight) - 1.0) > 0.000001
    ) allocation_problems;
    '''
)

bridge_checks["passed"] = bridge_checks["failed_rows"].eq(0)
display(bridge_checks)

if not bridge_checks["passed"].all():
    raise RuntimeError("Bridge integrity failed.")

print("Bridge integrity and allocation checks passed.")


,check_name,failed_rows,passed
0,bridge -> dim_account,0,True
1,bridge -> dim_customer,0,True
2,duplicate bridge account-customer pair,0,True
3,account allocation weights sum to 1,0,True


Bridge integrity and allocation checks passed.


## 14. Bank-Level KPI Sanity Checks

These checks do not enforce a target business performance level. They simply verify that the metrics are mathematically sensible for reporting.


In [14]:
kpi_summary = query_dataframe(
    '''
    SELECT
        COUNT(*) AS transaction_count,
        ROUND(SUM(amount), 2) AS total_transaction_value,
        ROUND(AVG(amount), 2) AS average_transaction_value,
        COUNT(*) FILTER (
            WHERE status = 'SUCCESSFUL'
        ) AS successful_transactions,
        COUNT(*) FILTER (
            WHERE status = 'FAILED'
        ) AS failed_transactions,
        COUNT(*) FILTER (
            WHERE status = 'REVERSED'
        ) AS reversed_transactions,
        ROUND(
            100.0
            * COUNT(*) FILTER (WHERE status = 'SUCCESSFUL')
            / NULLIF(COUNT(*), 0),
            2
        ) AS success_rate_pct,
        MIN(transaction_timestamp) AS first_transaction,
        MAX(transaction_timestamp) AS last_transaction
    FROM marts.fact_transactions;
    '''
)

display(kpi_summary)

kpi_row = kpi_summary.iloc[0]

kpi_checks = pd.DataFrame(
    [
        {
            "check": "transaction count is positive",
            "passed": kpi_row["transaction_count"] > 0,
        },
        {
            "check": "total transaction value is positive",
            "passed": kpi_row["total_transaction_value"] > 0,
        },
        {
            "check": "average transaction value is positive",
            "passed": kpi_row["average_transaction_value"] > 0,
        },
        {
            "check": "status counts equal transaction count",
            "passed": (
                kpi_row["successful_transactions"]
                + kpi_row["failed_transactions"]
                + kpi_row["reversed_transactions"]
                == kpi_row["transaction_count"]
            ),
        },
        {
            "check": "success rate is between 0 and 100",
            "passed": 0 <= kpi_row["success_rate_pct"] <= 100,
        },
        {
            "check": "transaction date range is valid",
            "passed": kpi_row["first_transaction"] <= kpi_row["last_transaction"],
        },
    ]
)

display(kpi_checks)

if not kpi_checks["passed"].all():
    raise RuntimeError("One or more KPI sanity checks failed.")

print("KPI sanity checks passed.")


,transaction_count,total_transaction_value,average_transaction_value,successful_transactions,failed_transactions,reversed_transactions,success_rate_pct,first_transaction,last_transaction
0,50000,128643291.50,2572.87,48235,1250,515,96.47,2018-01-26 00:57:14,2026-08-31 23:59:05


,check,passed
0,transaction count is positive,True
1,total transaction value is positive,True
2,average transaction value is positive,True
3,status counts equal transaction count,True
4,success rate is between 0 and 100,True
5,transaction date range is valid,True


KPI sanity checks passed.


## 15. KPI Distribution by Channel

A reporting-ready mart should contain sensible channel-level activity.


In [15]:
channel_kpis = query_dataframe(
    '''
    SELECT
        c.channel_code,
        c.channel_name,
        COUNT(*) AS transaction_count,
        ROUND(SUM(f.amount), 2) AS transaction_value,
        ROUND(
            100.0
            * COUNT(*) FILTER (WHERE f.status = 'SUCCESSFUL')
            / NULLIF(COUNT(*), 0),
            2
        ) AS success_rate_pct
    FROM marts.fact_transactions f
    JOIN marts.dim_channel c
        ON c.channel_key = f.channel_key
    GROUP BY c.channel_code, c.channel_name
    ORDER BY transaction_count DESC;
    '''
)

display(channel_kpis)

if channel_kpis.empty:
    raise RuntimeError("No channel KPI rows were produced.")

if (channel_kpis["transaction_count"] <= 0).any():
    raise RuntimeError("A reporting channel has no transaction activity.")

print("Channel KPI distribution looks valid.")


,channel_code,channel_name,transaction_count,transaction_value,success_rate_pct
0,CARD,Card,22175,8581697.89,96.51
1,APP,Mobile App,14595,83146338.50,96.52
2,ATM,ATM,13230,36915255.11,96.36


Channel KPI distribution looks valid.


## 16. Customer Allocation Reconciliation

Joint-account transaction values are allocated across customers using the bridge.

The total allocated value should reconcile back to the original fact-table transaction value, allowing only a very small rounding tolerance.


In [16]:
allocation_reconciliation = query_dataframe(
    '''
    WITH fact_total AS (
        SELECT
            SUM(amount) AS total_fact_value
        FROM marts.fact_transactions
    ),
    allocated_total AS (
        SELECT
            SUM(f.amount * b.allocation_weight) AS total_allocated_value
        FROM marts.fact_transactions f
        JOIN marts.bridge_account_customer b
            ON b.account_key = f.account_key
    )
    SELECT
        ROUND(total_fact_value, 2) AS total_fact_value,
        ROUND(total_allocated_value, 2) AS total_allocated_value,
        ROUND(
            ABS(total_fact_value - total_allocated_value),
            2
        ) AS absolute_difference
    FROM fact_total
    CROSS JOIN allocated_total;
    '''
)

display(allocation_reconciliation)

allocation_difference = float(
    allocation_reconciliation.iloc[0]["absolute_difference"]
)

if allocation_difference > 0.01:
    raise RuntimeError(
        "Customer allocation does not reconcile to fact transaction value."
    )

print("Customer allocation reconciles to bank-level transaction value.")


,total_fact_value,total_allocated_value,absolute_difference
0,128643291.50,128643291.50,0.00


Customer allocation reconciles to bank-level transaction value.


## 17. Rejected-Record Audit Review

Bad source rows should remain visible and explainable in the audit layer.


In [17]:
rejection_summary = query_dataframe(
    '''
    SELECT
        source_name,
        COUNT(*) AS rejected_rows
    FROM audit.rejected_records
    GROUP BY source_name
    ORDER BY source_name;
    '''
)

display(rejection_summary)

rejection_reason_summary = query_dataframe(
    '''
    SELECT
        source_name,
        rejection_reason,
        COUNT(*) AS rejected_rows
    FROM audit.rejected_records
    GROUP BY source_name, rejection_reason
    ORDER BY source_name, rejected_rows DESC, rejection_reason;
    '''
)

display(rejection_reason_summary)

if rejection_summary.empty:
    raise RuntimeError(
        "No rejected records were found. The synthetic source dataset "
        "was designed to contain controlled quality issues."
    )

print("Rejected-record auditing is populated and queryable.")


,source_name,rejected_rows
0,accounts,3
1,customer_accounts,5
2,customers,3
3,transactions,10


,source_name,rejection_reason,rejected_rows
0,accounts,account_type is not an allowed value,1
1,accounts,closed_date is earlier than opened_date,1
2,accounts,duplicate account_id; first occurrence retained,1
3,customer_accounts,account already has a PRIMARY holder; first PR...,1
4,customer_accounts,account_id does not reference an accepted stag...,1
5,customer_accounts,customer_id does not reference an accepted sta...,1
6,customer_accounts,duplicate customer-account relationship; first...,1
7,customer_accounts,holder_role is not an allowed value,1
8,customers,customer_since_date is not a valid date,1
9,customers,customer_status is not an allowed value,1


Rejected-record auditing is populated and queryable.


## 18. Consolidated Quality Gate

This final table brings together the major checks used to decide whether the database is ready for reporting.


In [18]:
quality_gate_rows = []

def add_results(category: str, dataframe: pd.DataFrame, name_column: str):
    for _, row in dataframe.iterrows():
        quality_gate_rows.append(
            {
                "category": category,
                "check": row[name_column],
                "passed": bool(row["passed"]),
            }
        )


add_results(
    "Raw to staging",
    raw_to_staging.rename(columns={"source_name": "check"}),
    "check",
)

add_results(
    "Staging keys",
    staging_key_checks,
    "check_name",
)

add_results(
    "Staging relationships",
    staging_fk_checks,
    "check_name",
)

add_results(
    "Staging business rules",
    staging_business_rules,
    "check_name",
)

add_results(
    "Staging domains",
    domain_checks,
    "check_name",
)

add_results(
    "Staging to marts",
    staging_to_mart,
    "check_name",
)

add_results(
    "Dimension keys",
    dimension_key_checks,
    "check_name",
)

add_results(
    "Fact quality",
    fact_checks,
    "check_name",
)

add_results(
    "Bridge quality",
    bridge_checks,
    "check_name",
)

for _, row in kpi_checks.iterrows():
    quality_gate_rows.append(
        {
            "category": "KPI sanity",
            "check": row["check"],
            "passed": bool(row["passed"]),
        }
    )

quality_gate = pd.DataFrame(quality_gate_rows)

display(quality_gate)

passed_checks = int(quality_gate["passed"].sum())
total_checks = len(quality_gate)
failed_checks = total_checks - passed_checks

print(f"Passed checks: {passed_checks}/{total_checks}")
print(f"Failed checks: {failed_checks}")

if failed_checks:
    raise RuntimeError(
        f"Database is not reporting-ready: {failed_checks} quality checks failed."
    )

print("DATABASE QUALITY GATE: PASSED")


,category,check,passed
0,Raw to staging,accounts,True
1,Raw to staging,customer_accounts,True
2,Raw to staging,customers,True
3,Raw to staging,transactions,True
4,Staging keys,staging.customers.customer_id,True
5,Staging keys,staging.accounts.account_id,True
6,Staging keys,"staging.customer_accounts(customer_id, account...",True
7,Staging keys,staging.transactions.transaction_id,True
8,Staging relationships,customer_accounts -> customers,True
9,Staging relationships,customer_accounts -> accounts,True


Passed checks: 50/50
Failed checks: 0
DATABASE QUALITY GATE: PASSED


## 19. Reporting Readiness Conclusion

The end-to-end pipeline has now been validated:

```text
CSV
 ↓
raw
 ↓
staging
 ↓
marts
 ↓
Metabase
```

The quality gate confirms:

- every source row is accounted for
- staging keys are unique
- staging relationships are valid
- business rules are enforced
- categorical domains are controlled
- current staging records are represented in marts
- fact-table grain is preserved
- dimensional foreign keys resolve
- Customer–Account bridge allocation is correct
- SCD Type 1 design is consistent
- KPI calculations are mathematically sensible
- rejected source records remain auditable

### Pipeline status

```text
01 Source profiling and validation   ✅
02 Raw ingestion                     ✅
03 Staging transformation            ✅
04 Dimensional mart build            ✅
05 End-to-end data quality checks    ✅
```

### Next section

The data pipeline is now ready for the **Metabase reporting layer**.

The next work is to design a small banking dashboard using the mart tables, with approximately 5–7 useful KPIs and visualisations.
